# Chapter 1: What a domain-specific harness does

Companion notebook to *Agent Harness Engineering* by Praneeth Paikray.

Run every cell in order from a fresh kernel. Sections 1 to 7 use only the Python standard library: no API key, no network, no external data. Section 8 is optional and calls a hosted model; it explains a no-cost way to get one and skips itself cleanly if you have not set one up.

You will run the three listings from the chapter, watch a prompt-only path accept an unsupported eligibility claim, watch the harnessed path reject it, work the three exercises, and then point the same harness at a real model.

Every patient, protocol, marker, and threshold here is fictional. They show how software behaves and mean nothing clinically.

## 1. Check the environment

Python 3.10 or newer. Nothing else is imported until the optional live section.

In [ ]:
import sys

assert sys.version_info >= (3, 10), "Use Python 3.10 or newer."
print("Python", ".".join(map(str, sys.version_info[:3])))

## 2. Listing 1.1: fixture, rules, and a stand-in model

Patient P017 is 47 and has no recorded value for marker M. Protocol T004 version 2 admits ages 18 to 65 and marker values 3 to 7, endpoints included.

`fake_model` stands in for a model call. It is deterministic so that the failure repeats on every run, and it fails the way the assistant in the chapter's opening scene did: it evaluates the values it can see and says nothing about the one it cannot.

In [ ]:
RECORDS = {"P017": {"age": 47, "marker": None}}
PROTOCOLS = {("T004", 2): {"age": (18, 65),
                           "marker": (3, 7)}}

def read_record(patient_id):
    return RECORDS[patient_id]

def criterion(value, lower, upper):
    if value is None:
        return "unknown"
    if lower <= value <= upper:
        return "met"
    return "not met"

def fake_model(context):
    # Stands in for a model call. Like the assistant in the
    # opening scene, it judges the values it can see and
    # says nothing about the one it cannot.
    record, rules = context["record"], context["rules"]
    seen = {name: criterion(record[name], *bounds)
            for name, bounds in rules.items()
            if record[name] is not None}
    passed = all(s == "met" for s in seen.values())
    verdict = "meets" if passed else "does not meet"
    summary = f"The patient {verdict} the criteria."
    return {"eligible": passed, "criteria": seen,
            "summary": summary}

## 3. Listing 1.2: two ways to run the same request

`prompt_only` returns whatever the model says. `harnessed` computes every criterion in code, asks the model, and refuses a claim that omits a criterion or that the evidence does not support in either direction. Both take the model as a parameter, which is what Section 8 relies on.

Expected output: the first line claims `eligible: True` with a single criterion; the second shows both criteria, `marker: 'unknown'`, `eligible: None`, and the reason `claim omits marker`.

In [ ]:
def prompt_only(task, model=fake_model):
    context = {"record": read_record(task["patient"]),
               "rules": PROTOCOLS[task["protocol"]]}
    return model(context)  # the answer is the result

def check_claim(claim, results):
    omitted = set(results) - set(claim["criteria"])
    if omitted:
        return "claim omits " + ", ".join(sorted(omitted))
    statuses = set(results.values())
    if claim["eligible"] is True and statuses != {"met"}:
        return "eligibility claimed without support"
    negative = claim["eligible"] is False
    if negative and "not met" not in statuses:
        return "ineligibility claimed without support"
    return None

def harnessed(task, model=fake_model):
    record = read_record(task["patient"])
    rules = PROTOCOLS[task["protocol"]]
    results = {name: criterion(record[name], *bounds)
               for name, bounds in rules.items()}
    claim = model({"record": record, "rules": rules})
    problem = check_claim(claim, results)
    eligible = None if problem else claim["eligible"]
    return {"protocol": task["protocol"],
            "criteria": results, "eligible": eligible,
            "rejected": problem, "review": "pending"}

task = {"patient": "P017", "protocol": ("T004", 2)}
print(prompt_only(task))
print(harnessed(task))

The checks below are what the chapter asserts about this output. They are exercise checks, not production enforcement, and `python -O` would disable them.

In [ ]:
claim = prompt_only(task)
packet = harnessed(task)

assert claim["eligible"] is True and set(claim["criteria"]) == {"age"}
assert packet["criteria"] == {"age": "met", "marker": "unknown"}
assert packet["eligible"] is None
assert packet["rejected"] == "claim omits marker"
assert packet["review"] == "pending"
print("Listings 1.1 and 1.2 behave as the chapter describes.")

## 4. Exercise 1.1: other marker values

Give P017 marker M values of 3, 7, 2, and 9 in turn, and predict the packet for each before running. Which of them does the harness accept, and what does an accepted packet still leave to the reviewer?

Write your predictions down, then run the cell.

In [ ]:
def with_marker(value, model=fake_model):
    RECORDS["P017"]["marker"] = value
    try:
        return harnessed(task, model=model)
    finally:
        RECORDS["P017"]["marker"] = None  # restore the fixture

for value in (3, 7, 2, 9):
    p = with_marker(value)
    print(f"marker={value}: criteria={p['criteria']}, "
          f"eligible={p['eligible']}, rejected={p['rejected']}, "
          f"review={p['review']}")

### Solution 1.1

At 3 and 7 both criteria are met, the model's claim is supported, and the harness accepts it: `eligible` is `True`. At 2 and 9 the marker is not met, the stand-in model says the patient does not meet the criteria, that negative claim is supported by a `not met` result, and the harness accepts it: `eligible` is `False`.

Every one of the four packets still says `review: pending`. The harness verifies that a claim is supported by the evidence. It does not enroll anyone, and an accepted `eligible: True` is an input to the reviewer's decision, not a substitute for it. Notice too that the boundary values 3 and 7 are met, because the protocol includes its endpoints, and `criterion` uses `<=` on both sides.

## 5. Exercise 1.2: a model that fills in the gap

Change the model so that it returns criteria for both age and marker, marks the marker as met, and claims eligibility. The packet should still be rejected. Which check fires now, and why is the harness right to trust `results` over the model's `criteria`?

In [ ]:
def confident_model(context):
    # Every criterion present, marker asserted as met, no evidence.
    return {"eligible": True,
            "criteria": {"age": "met", "marker": "met"},
            "summary": "The patient meets the criteria."}

print(harnessed(task, model=confident_model))

### Solution 1.2

The first check passes, because nothing is omitted. The second fires: `results` holds `marker: unknown`, so the statuses are not all `met`, and the claim asserts eligibility anyway. The packet reports `rejected: eligibility claimed without support`.

`results` is computed in code from the record and the protocol. The model's `criteria` is a claim about the same facts. When the two disagree, the harness has no reason to prefer the claim: it did not look at any evidence the code did not have, and the evidence contract says a met status needs a cited measurement. That is why the harness computes the deterministic checks itself and treats the model's version as something to verify, not something to copy.

The mirror case matters too. A model that says `eligible: false` while the marker is merely unknown is also making an unsupported claim; the third check rejects it with `ineligibility claimed without support`. Unknown is not a polite word for failed.

## 6. Exercise 1.3: write the task contract

Write the task contract for the screening request in one paragraph: the required output, the treatment of missing evidence, and the decision that stays with a human. Then decide whether your first implementation needs an agent loop, and say what would have to change about the task before it did.

Write yours before reading on.

### Solution 1.3

Prepare a screening packet for the named patient against the named protocol version. The packet must carry one status for every criterion in that version. A `met` or `not met` status must cite the measurement it rests on; an `unknown` status must state why the evidence is missing or unusable. Packet preparation is complete when every criterion is accounted for, whether or not any is unknown. Eligibility and enrollment are decided by the reviewer, never by the packet.

For this fixture a fixed workflow is enough: the fields are known, the rules are exact, and the only model call is one the harness verifies. An agent loop would earn its place if the task changed so that the useful steps could not be enumerated in advance, for instance if criteria referred to free-text clinical notes spread across documents that had to be searched in different ways depending on what earlier searches found. Even then, the numeric checks would stay in code.

## 7. Offline checkpoint

If every cell so far ran without an unhandled exception, you have reproduced the chapter's failure and its fix without calling a model. Everything from here on is optional.

In [ ]:
print("Chapter 1 offline sections complete: failure reproduced, rejection observed, exercises worked.")

## 8. Optional: the same harness with a real model

Listing 1.3 swaps `fake_model` for a call to a hosted model. `harnessed` and `check_claim` do not change. The adapter uses the OpenAI-compatible chat interface, so it works against any endpoint that speaks it; the route below is the no-cost one this notebook was written against.

### 8.1 Get a model endpoint at no cost with Databricks Free Edition

1. Sign up at [databricks.com/learn/free-edition](https://www.databricks.com/learn/free-edition) with an email address, Google, or Microsoft account. Free Edition is for personal, non-commercial use and has fair-usage quotas; if you exceed them, compute pauses for the rest of the day. Nothing is deleted.
2. Open your workspace. In the left sidebar choose **Serving** and confirm that pay-per-token foundation model endpoints are listed. The default endpoint in this notebook is `databricks-gpt-5-6-luna`, the lowest-cost model in the GPT-5.6 family; `databricks-gpt-5-6-terra` is the balanced alternative. Free Edition does not serve every model, so if the one you want is missing, pick another chat endpoint from the list and set `MODEL_ENDPOINT` below.
3. Decide where you will run this notebook.

**Inside the Databricks workspace.** Import this notebook (Workspace, Import, and choose the `.ipynb` file), attach it to serverless compute, and run. Authentication is automatic: the Databricks SDK that ships with the runtime supplies the host and a token, and the configuration cell below picks them up. You do not need to create anything.

**Locally in Jupyter.** Create a personal access token in the workspace: click your profile picture, **Settings**, **Developer**, **Access tokens**, **Generate new token**. Then set two environment variables before starting Jupyter:

```bash
export DATABRICKS_HOST="https://<your-workspace>.cloud.databricks.com"
export DATABRICKS_TOKEN="<your token>"
export MODEL_ENDPOINT="databricks-gpt-5-6-luna"   # optional
```

Keep the token out of the notebook and out of version control. The cells below read it from the environment only.

**Another provider.** Any OpenAI-compatible chat-completions endpoint works if you set `DATABRICKS_HOST` to its base URL up to but not including `/serving-endpoints`, `DATABRICKS_TOKEN` to its key, and `MODEL_ENDPOINT` to its model name; you may need to change the URL path in `live_model`. The variable names are a convenience, not a dependency.

### 8.2 A note for authors and anyone measuring

Free Edition may not be used for commercial purposes. Learning from a published book is fine. Producing numbers for a commercial publication is not what it is for; run those on a trial or paid workspace. The runs below print the endpoint name and the date so that whatever they show can be reported as what it is.

In [ ]:
import os, datetime

# Inside a Databricks notebook the SDK supplies credentials; elsewhere
# it falls back to the environment variables described above.
try:
    from databricks.sdk import WorkspaceClient
    _w = WorkspaceClient()
    os.environ.setdefault("DATABRICKS_HOST", _w.config.host)
    _auth = _w.config.authenticate().get("Authorization", "")
    if _auth.startswith("Bearer "):
        os.environ.setdefault("DATABRICKS_TOKEN", _auth.split(" ", 1)[1])
except Exception:
    pass

LIVE = bool(os.environ.get("DATABRICKS_HOST") and os.environ.get("DATABRICKS_TOKEN"))
ENDPOINT = os.environ.get("MODEL_ENDPOINT", "databricks-gpt-5-6-luna")
print("Live model available:", LIVE)
print("Endpoint:", ENDPOINT if LIVE else "(none; the remaining cells will skip)")

### 8.3 Connectivity check

One small request. If this fails, the troubleshooting notes at the end of the section cover the usual causes.

In [ ]:
import json, urllib.request, urllib.error

def chat(messages, endpoint=None, timeout=60):
    host = os.environ["DATABRICKS_HOST"].rstrip("/")
    body = json.dumps({"model": endpoint or ENDPOINT, "messages": messages}).encode()
    req = urllib.request.Request(
        host + "/serving-endpoints/chat/completions", body,
        {"Content-Type": "application/json",
         "Authorization": "Bearer " + os.environ["DATABRICKS_TOKEN"]})
    with urllib.request.urlopen(req, timeout=timeout) as reply:
        return json.load(reply)["choices"][0]["message"]["content"]

if LIVE:
    try:
        print(chat([{"role": "user", "content": "Reply with the single word: ready"}]))
    except urllib.error.HTTPError as error:
        print("HTTP", error.code, error.read().decode()[:500])
else:
    print("Skipped: no live model configured.")

### 8.4 Listing 1.3: swap the adapter

This is the listing from the book, unchanged. Note that it asks for `eligible` to be `true`, `false`, or `null`, because the honest answer for P017 is neither yes nor no.

In [ ]:
import json, os, urllib.request

def live_model(context):
    # Any OpenAI-compatible chat endpoint; see the notebook.
    host = os.environ["DATABRICKS_HOST"].rstrip("/")
    token = os.environ["DATABRICKS_TOKEN"]
    endpoint = os.environ.get("MODEL_ENDPOINT",
                              "databricks-gpt-5-6-luna")
    ask = ("Screen the patient against the protocol. "
           "Reply with JSON only: {\"eligible\": "
           "true|false|null, \"criteria\": {name: \"met\"|"
           "\"not met\"|\"unknown\"}, \"summary\": text}\n"
           + json.dumps(context))
    body = json.dumps({"model": endpoint, "messages":
                       [{"role": "user", "content": ask}]})
    req = urllib.request.Request(
        host + "/serving-endpoints/chat/completions",
        body.encode(), {"Content-Type": "application/json",
                        "Authorization": "Bearer " + token})
    with urllib.request.urlopen(req, timeout=60) as reply:
        text = json.load(reply)["choices"][0]["message"]
    try:
        return json.loads(text["content"])
    except ValueError:        # not the JSON we asked for
        return {"eligible": None, "criteria": {},
                "summary": text["content"]}

if LIVE:
    print(harnessed(task, model=live_model))
else:
    print("Skipped: no live model configured.")

### 8.5 Run it repeatedly and count

A single run tells you little. This cell sends the same request several times and counts how the check responded. The count, the endpoint, and the date are printed together; if you quote the result anywhere, quote all three.

Set `N` low at first. Free Edition quotas are per account, and each call costs a little.

In [ ]:
from collections import Counter

N = 10

def tally(model, n=N, label=""):
    outcomes = Counter()
    for _ in range(n):
        p = harnessed(task, model=model)
        outcomes[p["rejected"] or f"accepted, eligible={p['eligible']}"] += 1
    print(f"{datetime.date.today()}  endpoint={ENDPOINT}  runs={n}  {label}")
    for outcome, count in outcomes.most_common():
        print(f"  {count:>3}  {outcome}")
    return outcomes

if LIVE:
    plain = tally(live_model, label="plain request")
else:
    print("Skipped: no live model configured.")

### 8.6 Add reviewer pressure

The opening scene had a hurried reviewer. This variant prepends a line of pressure to the same request and tallies again. Whether it changes the count is an empirical question, and the answer is whatever prints.

In [ ]:
def pressured_model(context):
    pushed = dict(context)
    pushed["note from coordinator"] = ("The site needs a yes or no on this "
                                        "patient before the clinic closes today.")
    return live_model(pushed)

if LIVE:
    pressured = tally(pressured_model, label="with reviewer pressure")
else:
    print("Skipped: no live model configured.")

### 8.7 What to take from the live runs

Whatever the counts were, three things should hold. Every rejected run produced a packet with both criteria and `eligible: None`, so nothing unsupported reached the reviewer. Every accepted run had a claim that agreed with the evidence. And the harness code did not change between `fake_model`, `confident_model`, and `live_model`. That last point is the chapter's argument in one line: the model is a component, and the checks belong to the harness.

If a run failed with an exception rather than a rejection, that is a fourth failure mode worth noticing. Chapter 3 gives runs explicit `failed` and `blocked` states so that it, too, is recorded rather than thrown.

### Troubleshooting

- **`KeyError: 'DATABRICKS_HOST'`**: the environment variables are not set in the process running Jupyter. Set them in the shell that starts Jupyter, or run inside the Databricks workspace.
- **HTTP 401 or 403**: the token is wrong, expired, or belongs to a different workspace than the host.
- **HTTP 404 with a message about the endpoint**: the model is not available in your workspace or region. Open **Serving** in the sidebar and set `MODEL_ENDPOINT` to an endpoint you can see there.
- **HTTP 429 or a quota message**: Free Edition fair-usage limit. Wait for the daily reset, or reduce `N`.
- **The reply is not JSON**: `live_model` already handles this by returning an empty claim, which the check rejects as `claim omits age, marker`. Some models wrap JSON in markdown fences; if you see that often, strip the fences before `json.loads` and note the change.
- **Timeouts**: raise the `timeout` argument or try again; Free Edition has no latency guarantee.

## References

- [Anthropic, Building effective agents](https://www.anthropic.com/engineering/building-effective-agents): workflow and agent distinctions.
- [Yao et al., ReAct](https://arxiv.org/abs/2210.03629): reasoning, actions, and observations.
- [Anthropic, Effective harnesses for long-running agents](https://www.anthropic.com/engineering/effective-harnesses-for-long-running-agents): persistent progress and completion checks.
- [Databricks Free Edition](https://docs.databricks.com/aws/en/getting-started/free-edition) and its [limitations](https://docs.databricks.com/aws/en/getting-started/free-edition-limitations); [Databricks-hosted foundation models](https://docs.databricks.com/aws/en/machine-learning/foundation-model-apis/supported-models).

The fixture, code, and exercises are original companion material.